---
jupytext:
  text_representation:
    format_name: myst
kernelspec:
  display_name: Python 3
  language: python
  name: python3
---
# Lesson 3 Ã¢â‚¬â€ Visualising Streamflow Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mr-pablinho/drops-course/blob/main/book/01-python-for-hydrology/03-visualization.ipynb)

**Learning objectives**  
By the end of this lesson you will be able to:
- [ ] Build multi-panel Matplotlib figures with shared axes and formatted tick labels
- [ ] Plot a **Flow Duration Curve (FDC)** on a log-log scale and read off exceedance probabilities
- [ ] Create monthly **boxplots** to reveal seasonal discharge variability
- [ ] Build an interactive Plotly hydrograph with a range slider

In [ ]:
# Install course dependencies when running on Google Colab
import sys
if 'google.colab' in sys.modules:
    import subprocess
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', 'https://raw.githubusercontent.com/mr-pablinho/drops-course/main/requirements-colab.txt'
    ], check=True)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import plotly.graph_objects as go

In [ ]:
CI = os.environ.get('HYDRO_ML_CI', '0') == '1'

if CI:
    df_raw = pd.read_parquet('../../data/samples/usgs_06803495_daily_2000_2015.parquet')
else:
    import dataretrieval.nwis as nwis
    raw, _ = nwis.get_dv(
        sites='06803495',
        parameterCd='00060',
        start='2000-01-01',
        end='2015-12-31',
    )
    df_raw = raw.reset_index()

df = (
    df_raw
    .rename(columns={'00060_Mean': 'discharge_cfs'})
    .assign(datetime=lambda d: pd.to_datetime(d['datetime']))
    .set_index('datetime')
    .sort_index()
    [['discharge_cfs']]
    .pipe(lambda d: d[d['discharge_cfs'] > 0])
    .dropna()
)

print(f'Loaded {len(df):,} daily records from {df.index.min().date()} to {df.index.max().date()}')

## Introduction

In hydrology, the right chart is often the *only* way to see what is happening:
a table of 5,844 discharge values conceals the 2011 flood completely, but a hydrograph
makes it unmissable in under a second.

This lesson covers four workhorse visualisations that every hydrologist uses:

| Chart type | What it answers |
|---|---|
| Multi-panel hydrograph | When did extreme events occur? |
| Flow Duration Curve | How often does flow exceed a given level? |
| Monthly boxplot | How variable is flow within each month? |
| Interactive Plotly chart | How do I explore the record interactively? |

## 1. Multi-Panel Hydrograph

A single large figure with two vertically stacked panels Ã¢â‚¬â€ the full record on top,
a zoomed event on the bottom Ã¢â‚¬â€ is the standard format for presenting a streamflow
record alongside a flood event in publications and reports.

`plt.subplots(nrows, ncols, figsize, sharex)` creates a figure with a grid of `Axes`
objects. `sharex=True` links the x-axis zoom across panels, which is not useful here
because the two panels cover different time spans Ã¢â‚¬â€ so we keep them independent.

In [ ]:
flood = df['2010-10-01':'2012-03-31']
peak_idx = flood['discharge_cfs'].idxmax()
peak_val = flood.loc[peak_idx, 'discharge_cfs']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))

# Ã¢â‚¬â€ Top panel: full 15-year record Ã¢â‚¬â€
ax1.fill_between(df.index, df['discharge_cfs'] / 1_000,
                 alpha=0.20, color='steelblue')
ax1.plot(df.index, df['discharge_cfs'] / 1_000,
         color='steelblue', lw=0.8, alpha=0.9)
ax1.axvspan(pd.Timestamp('2010-10-01'), pd.Timestamp('2012-03-31'),
            color='gold', alpha=0.25, label='Flood window (detail below)')
ax1.set_ylabel('Discharge (kcfs)', fontsize=11)
ax1.set_title('Missouri River at Omaha Ã¢â‚¬â€ Daily Mean Discharge 2000Ã¢â‚¬â€œ2015', fontsize=11)
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax1.legend(fontsize=10)

# Ã¢â‚¬â€ Bottom panel: flood event detail Ã¢â‚¬â€
ax2.fill_between(flood.index, flood['discharge_cfs'] / 1_000,
                 alpha=0.25, color='crimson')
ax2.plot(flood.index, flood['discharge_cfs'] / 1_000,
         color='crimson', lw=1.5)
ax2.axvline(peak_idx, color='darkred', ls='--', lw=1.8,
            label=f'Peak: {peak_val:,.0f} cfs ({peak_idx.date()})')
ax2.set_xlabel('Date', fontsize=11)
ax2.set_ylabel('Discharge (kcfs)', fontsize=11)
ax2.set_title('2011 Missouri River Flood Ã¢â‚¬â€ Oct 2010 to Mar 2012', fontsize=11)
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}k'))
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

## 2. Flow Duration Curve (FDC)

A **Flow Duration Curve** ranks every daily discharge value from highest to lowest and
plots each against its **exceedance probability** Ã¢â‚¬â€ the fraction of days on which flow
equalled or exceeded that value.

$$P(Q \geq q) = \frac{\text{rank}(q)}{n + 1}$$

Key reference points:

| Exceedance probability | Hydrological interpretation |
|---|---|
| < 5 % | Flood / very high flow |
| 20Ã¢â‚¬â€œ40 % | High-flow season |
| ~50 % | Median flow (Q50) |
| > 90 % | Low-flow / drought conditions |

Plotting on a **log-log scale** straightens FDCs for many rivers, making the
high-flow and low-flow tails visible simultaneously.

In [ ]:
q_sorted = np.sort(df['discharge_cfs'].values)[::-1]   # descending
n = len(q_sorted)
exceed_prob = np.arange(1, n + 1) / (n + 1)            # Weibull plotting position

# Reference percentiles to annotate
refs = {5: 'Q5 (flood)', 50: 'Q50 (median)', 90: 'Q90 (low flow)'}

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(exceed_prob * 100, q_sorted / 1_000,
        color='steelblue', lw=1.8, label='Daily FDC (2000Ã¢â‚¬â€œ2015)')

for pct, label in refs.items():
    idx = int(pct / 100 * n)
    qval = q_sorted[idx]
    ax.annotate(
        f'{label}\n{qval:,.0f} cfs',
        xy=(pct, qval / 1_000),
        xytext=(pct + 3, qval / 1_000 * 1.6),
        arrowprops=dict(arrowstyle='->', color='navy'),
        fontsize=8.5, color='navy',
    )

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Exceedance probability (%)', fontsize=11)
ax.set_ylabel('Discharge (kcfs)', fontsize=11)
ax.set_title('Flow Duration Curve Ã¢â‚¬â€ Missouri River at Omaha (2000Ã¢â‚¬â€œ2015)', fontsize=11)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.3g}%'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}k'))
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3. Monthly Boxplots: Seasonal Variability

Monthly means (Lesson 2) reveal the seasonal *centre* of the distribution, but
hide **spread and skew**. A **boxplot** shows median, interquartile range (IQR),
and outliers simultaneously.

For streamflow Ã¢â‚¬â€ which is always positive and often log-normally distributed Ã¢â‚¬â€
plotting on a **log y-scale** prevents the 2011 outliers from compressing the
boxes for other months into a thin line at the bottom of the figure.

In [ ]:
month_data = [df.loc[df.index.month == m, 'discharge_cfs'].values
              for m in range(1, 13)]
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(
    [arr / 1_000 for arr in month_data],
    labels=month_names,
    patch_artist=True,
    medianprops=dict(color='navy', lw=2),
    flierprops=dict(marker='o', ms=3, alpha=0.4, color='steelblue'),
    boxprops=dict(facecolor='lightsteelblue', alpha=0.7),
)

ax.set_yscale('log')
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Discharge (kcfs, log scale)', fontsize=11)
ax.set_title('Missouri River Ã¢â‚¬â€ Monthly Discharge Distribution 2000Ã¢â‚¬â€œ2015', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}k' if x >= 1 else f'{x:.2f}k'))
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Interactive Hydrograph with Plotly

Static Matplotlib figures are ideal for publications and reports. For **exploration** Ã¢â‚¬â€
panning, zooming, hovering over individual dates Ã¢â‚¬â€ **Plotly** provides interactive
HTML-based charts that render directly in Jupyter and JupyterBook.

The `rangeslider` widget on the x-axis lets you scrub through the full record while
keeping a zoomed detail view in the main panel.

In [ ]:
roll30 = df['discharge_cfs'].rolling(30, min_periods=20).mean()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['discharge_cfs'] / 1_000,
    mode='lines',
    name='Daily discharge',
    line=dict(color='steelblue', width=0.8),
    opacity=0.7,
    hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} kcfs<extra></extra>',
))

fig.add_trace(go.Scatter(
    x=roll30.index,
    y=roll30 / 1_000,
    mode='lines',
    name='30-day mean',
    line=dict(color='navy', width=2),
    hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} kcfs<extra></extra>',
))

fig.update_layout(
    title='Missouri River at Omaha Ã¢â‚¬â€ Interactive Hydrograph 2000Ã¢â‚¬â€œ2015',
    xaxis=dict(
        title='Date',
        rangeslider=dict(visible=True),
        rangeselector=dict(buttons=[
            dict(count=1, label='1y', step='year', stepmode='backward'),
            dict(count=3, label='3y', step='year', stepmode='backward'),
            dict(step='all', label='All'),
        ]),
    ),
    yaxis=dict(title='Discharge (kcfs)'),
    height=450,
    template='simple_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
)

fig.show()

## Key Takeaways

- **Multi-panel figures** (`plt.subplots`) let you show context (full record) and
  detail (flood event) side by side without duplicating axes.
- The **Flow Duration Curve** on a log-log scale is the single most information-dense
  hydrograph: one curve communicates the entire discharge distribution across
  seven orders of magnitude.
- **Monthly boxplots on a log scale** reveal seasonal spread and outlier structure
  that bar charts of monthly means completely obscure.
- **Plotly** fills the gap between static figures and full dashboards: `rangeslider`
  and `rangeselector` are one-line additions that make a 15-year record explorable
  without any JavaScript.

## Exercise

**Try it yourself:**  
Overlay the Flow Duration Curves for **two individual years** Ã¢â‚¬â€ 2005 (a near-normal year)
and 2011 (the flood year) Ã¢â‚¬â€ on the same log-log plot, alongside the full-record FDC
from this lesson.

What does the 2011 curve look like compared to 2005 in the high-flow tail
(exceedance probability < 5 %)?

*Hint: filter `df` by year with `df['2005']` and `df['2011']`, then repeat the
sort/exceedance calculation for each subset.*

In [ ]:
# Solution Ã¢â‚¬â€ try on your own first!
def fdc(series):
    """Return (exceedance_pct, sorted_discharge_kcfs) for an FDC."""
    q = np.sort(series.dropna().values)[::-1]
    p = np.arange(1, len(q) + 1) / (len(q) + 1) * 100
    return p, q / 1_000

fig, ax = plt.subplots(figsize=(9, 5))

for label, color, data in [
    ('All years 2000Ã¢â‚¬â€œ2015', 'steelblue', df['discharge_cfs']),
    ('2005 (near-normal)',  'forestgreen', df['2005']['discharge_cfs']),
    ('2011 (flood year)',   'crimson',      df['2011']['discharge_cfs']),
]:
    p, q = fdc(data)
    ax.plot(p, q, lw=1.8, label=label, color=color)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Exceedance probability (%)', fontsize=11)
ax.set_ylabel('Discharge (kcfs, log scale)', fontsize=11)
ax.set_title('Flow Duration Curves Ã¢â‚¬â€ Full Record vs 2005 vs 2011', fontsize=11)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.3g}%'))
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}k' if x >= 1 else f'{x:.2f}k'))
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()